# 2. Policy Improvement
### Improves an Existing Policy

**Policy Improvement** derives a strictly better (or equal) policy $\pi'$ from the value function $V^\pi$ of a current policy $\pi$ by acting greedily with respect to the state-action values $q_\pi(s, a)$.

#### Mathematical Foundation:
1. **State-Action Value Calculation ($q_\pi$)**:
   $$q_\pi(s, a) = \sum_{s', r} P(s', r | s, a) \left[ r + \gamma V^\pi(s') \right]$$
2. **Greedy Policy Extraction**:
   $$\pi'(s) = \arg\max_a q_\pi(s, a)$$
3. **Policy Improvement Theorem**:
   For all states $s \in \mathcal{S}$:
   $$q_\pi(s, \pi'(s)) \ge V^\pi(s) \implies V^{\pi'}(s) \ge V^\pi(s)$$
   If $V^{\pi'}(s) = V^\pi(s)$ for all $s$, then $\pi$ and $\pi'$ are optimal policies $\pi^*$.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Step 2: Define a 4x4 GridWorld MDP Environment
# States: 16 grid cells (0 to 15), where (0,0) [state 0] and (3,3) [state 15] are terminal
# Actions: 0: UP, 1: RIGHT, 2: DOWN, 3: LEFT

class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.num_states = size * size
        self.num_actions = 4
        self.actions = ['UP', 'RIGHT', 'DOWN', 'LEFT']
        self.terminal_states = {0, self.num_states - 1}
        
    def get_next_state_and_reward(self, state, action):
        if state in self.terminal_states:
            return state, 0.0
            
        r, c = divmod(state, self.size)
        if action == 0:    # UP
            r = max(r - 1, 0)
        elif action == 1:  # RIGHT
            c = min(c + 1, self.size - 1)
        elif action == 2:  # DOWN
            r = min(r + 1, self.size - 1)
        elif action == 3:  # LEFT
            c = max(c - 1, 0)
            
        next_state = r * self.size + c
        reward = -1.0  # -1 step cost until terminal goal reached
        return next_state, reward

env = GridWorld()
print(f"GridWorld initialized with {env.num_states} states and {env.num_actions} actions.")


In [ ]:
# Step 3: Implement Policy Evaluation & Policy Improvement
def evaluate_policy(env, policy, gamma=1.0, theta=1e-4):
    V = np.zeros(env.num_states)
    while True:
        delta = 0.0
        V_new = np.copy(V)
        for s in range(env.num_states):
            if s in env.terminal_states:
                continue
            v_s = sum(
                policy[s][a] * (env.get_next_state_and_reward(s, a)[1] + gamma * V[env.get_next_state_and_reward(s, a)[0]])
                for a in range(env.num_actions)
            )
            V_new[s] = v_s
            delta = max(delta, abs(V[s] - V_new[s]))
        V = V_new
        if delta < theta:
            break
    return V

def policy_improvement(env, V, gamma=1.0):
    policy = np.zeros((env.num_states, env.num_actions))
    policy_stable = True
    
    for s in range(env.num_states):
        if s in env.terminal_states:
            continue
            
        action_values = np.zeros(env.num_actions)
        for a in range(env.num_actions):
            next_s, r = env.get_next_state_and_reward(s, a)
            action_values[a] = r + gamma * V[next_s]
            
        best_a = np.argmax(action_values)
        policy[s, best_a] = 1.0  # Deterministic greedy policy
        
    return policy


In [ ]:
# Step 4: One-Step Policy Improvement from Uniform Random Policy
random_policy = np.ones((env.num_states, env.num_actions)) / env.num_actions
V_old = evaluate_policy(env, random_policy)

improved_policy = policy_improvement(env, V_old)
V_new = evaluate_policy(env, improved_policy)

print("Average value under initial random policy:", np.mean(V_old))
print("Average value under 1-step improved policy:", np.mean(V_new))


In [ ]:
# Function to plot State-Value Heatmap and Policy Arrows
def plot_grid(V, policy=None, title="State-Value Grid"):
    fig, ax = plt.subplots(figsize=(6, 6))
    grid_v = V.reshape((4, 4))
    
    cax = ax.matshow(grid_v, cmap='Blues_r')
    fig.colorbar(cax)
    
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    
    for r in range(4):
        for c in range(4):
            s = r * 4 + c
            val_text = f"{V[s]:.1f}"
            if s in [0, 15]:
                val_text += "\n(Goal)"
            elif policy is not None:
                best_a = np.argmax(policy[s])
                val_text += f"\n{arrows[best_a]}"
                
            ax.text(c, r, val_text, ha='center', va='center', color='black', fontsize=11, weight='bold')
            
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    plt.title(title, pad=20)
    plt.show()


In [ ]:
# Step 6: Visualizing the Improved Policy
plot_grid(V_new, policy=improved_policy, title="Improved Policy and Resulting State-Values")


### Summary
- Takes $V^\pi$ and acts greedily to extract a superior policy $\pi'$.
- Guaranteed by the Policy Improvement Theorem to never degrade performance ($V^{\pi'}(s) \ge V^\pi(s)$).
